In [2]:
# %% [markdown]
# # Step 4: Create Complete Preprocessing Pipeline
# ## Memory-Optimized Version (No Display to Prevent Kernel Death)

# %%
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
import gc
import sys
import warnings
warnings.filterwarnings('ignore')

# %%
# Memory optimization settings
import torch
torch.set_num_threads(1)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# %%
class ImagePreprocessor:
    def __init__(self, target_size=(224, 224)):  # DenseNet121 expects 224x224
        self.target_size = target_size
    
    def preprocess_for_model(self, image):
        """
        Final preprocessing for model input
        """
        if isinstance(image, Image.Image):
            image = np.array(image)
        elif isinstance(image, torch.Tensor):
            image = image.cpu().numpy()
        
        # Ensure image is uint8 for resize
        if image.dtype != np.uint8:
            if image.max() <= 1.0:
                image = (image * 255).astype(np.uint8)
            else:
                image = image.astype(np.uint8)
        
        # Resize
        image = cv2.resize(image, self.target_size)
        
        # Normalize to [0,1]
        image = image.astype(np.float32) / 255.0
        
        return image
    
    def combine_with_features(self, image, fertilizer_amount, days):
        """
        Combine image with numerical features
        """
        features = np.array([fertilizer_amount, days])
        return image, features

# %%
# Add path to utils
utils_path = os.path.join(os.getcwd(), 'utils')
if utils_path not in sys.path:
    sys.path.append(utils_path)

# %%
# Helper function to get a test image from the project
def get_test_image():
    """Get the first image from static/uploads folder or create a dummy image"""
    # First, check static/uploads folder for existing images
    upload_folder = os.path.join(os.getcwd(), 'static', 'uploads')
    if os.path.exists(upload_folder):
        images = [f for f in os.listdir(upload_folder) 
                 if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif'))]
        if images:
            return os.path.join(upload_folder, images[0])
    
    # If no images found, check if there's a sample image in test_output
    test_output = os.path.join(os.getcwd(), 'test_output')
    if os.path.exists(test_output):
        images = [f for f in os.listdir(test_output) 
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if images:
            return os.path.join(test_output, images[0])
    
    # If no images found, return None (will use dummy image)
    return None

# %%
# Test complete pipeline with proper memory management (NO DISPLAY)
print("="*50)
print("Testing Complete Preprocessing Pipeline")
print("="*50)

try:
    # Import with error handling
    try:
        from utils.background_removal import BackgroundRemover
        print("✅ BackgroundRemover imported")
    except Exception as e:
        print(f"❌ Failed to import BackgroundRemover: {e}")
        BackgroundRemover = None
    
    try:
        from utils.edge_detection import EdgeDetector
        print("✅ EdgeDetector imported")
    except Exception as e:
        print(f"❌ Failed to import EdgeDetector: {e}")
        EdgeDetector = None
    
    # Initialize components - Using portable BackgroundRemover
    current_dir = os.getcwd()
    
    # Initialize background remover (portable version - no path needed)
    bg_remover = None
    if BackgroundRemover is not None:
        try:
            # The portable BackgroundRemover finds weights automatically
            print(f"\n🔄 Initializing portable BackgroundRemover...")
            bg_remover = BackgroundRemover()  # No path needed - auto-finds in u2net/weights/
            print("✅ BackgroundRemover initialized")
        except Exception as e:
            print(f"⚠️ Could not initialize BackgroundRemover: {e}")
            print("   Will use fallback mode")
            bg_remover = None
    else:
        print("⚠️ Skipping U2Net initialization - will use fallback")
    
    edge_detector = EdgeDetector() if EdgeDetector is not None else None
    preprocessor = ImagePreprocessor(target_size=(224, 224))
    
    # Get test image from project folders
    test_image_path = get_test_image()
    
    if test_image_path and os.path.exists(test_image_path):
        print(f"\n🔄 Processing image: {test_image_path}")
        
        # Read image and immediately resize to smaller size
        image = cv2.imread(test_image_path)
        
        if image is None:
            print("❌ Could not read image, creating dummy image for testing...")
            # Create a dummy image for testing
            image = np.zeros((800, 800, 3), dtype=np.uint8)
            image[:, :] = [100, 150, 200]  # Light blue background
            # Draw a green square in the center (simulating leaf)
            cv2.rectangle(image, (200, 200), (600, 600), (0, 255, 0), -1)
            print("   Using dummy image for testing")
        else:
            # Resize image to reasonable size before any processing
            height, width = image.shape[:2]
            if height > 800 or width > 800:
                scale = 800 / max(height, width)
                new_width = int(width * scale)
                new_height = int(height * scale)
                image = cv2.resize(image, (new_width, new_height))
                print(f"📏 Image resized to {new_width}x{new_height} for processing")
        
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Step 1: Background removal (with memory optimization)
        print("\n🔄 Step 1: Background removal...")
        if bg_remover is not None:
            try:
                # Using the portable background remover with memory optimization
                bg_removed, mask = bg_remover.remove_background(
                    image_rgb, 
                    target_size=(224, 224),  # Match model input size
                    max_size=800  # Memory optimization
                )
                print("   ✅ Background removal complete")
                print(f"   Background removed shape: {bg_removed.shape}")
                if mask is not None:
                    print(f"   Mask shape: {mask.shape}")
            except Exception as e:
                print(f"   ⚠️ Background removal failed: {e}")
                # Fallback: just resize
                bg_removed = cv2.resize(image_rgb, (224, 224))
                mask = None
        else:
            print("   ⚠️ Using fallback (no background removal)")
            bg_removed = cv2.resize(image_rgb, (224, 224))
            mask = None
        
        # Force garbage collection
        gc.collect()
        
        # Step 2: Edge detection
        print("\n🔄 Step 2: Edge detection...")
        if edge_detector is not None and bg_removed is not None:
            try:
                edge_image, edges = edge_detector.detect_edges(bg_removed)
                print("   ✅ Edge detection complete")
                print(f"   Edge detected shape: {edge_image.shape}")
            except Exception as e:
                print(f"   ⚠️ Edge detection failed: {e}")
                edge_image = bg_removed
                edges = None
        else:
            print("   ⚠️ Using fallback (no edge detection)")
            edge_image = bg_removed
        
        # Force garbage collection
        gc.collect()
        
        # Step 3: Final preprocessing
        print("\n🔄 Step 3: Final preprocessing...")
        processed = preprocessor.preprocess_for_model(edge_image)
        print(f"   ✅ Preprocessing complete - shape: {processed.shape}")
        print(f"   Final image range: [{processed.min():.2f}, {processed.max():.2f}]")
        print(f"   Final image dtype: {processed.dtype}")
        
        # Save images instead of displaying (to verify results)
        print("\n💾 Saving results to verify...")
        
        # Create output directory
        output_dir = os.path.join(current_dir, 'test_output')
        os.makedirs(output_dir, exist_ok=True)
        
        # Save intermediate images
        cv2.imwrite(os.path.join(output_dir, '1_original.jpg'), 
                   cv2.cvtColor(cv2.resize(image_rgb, (224, 224)), cv2.COLOR_RGB2BGR))
        
        if mask is not None:
            cv2.imwrite(os.path.join(output_dir, '2_mask.jpg'), mask)
        
        cv2.imwrite(os.path.join(output_dir, '3_background_removed.jpg'), 
                   cv2.cvtColor(cv2.resize(bg_removed, (224, 224)), cv2.COLOR_RGB2BGR))
        
        # Save edge detected image
        cv2.imwrite(os.path.join(output_dir, '3_edge_detected.jpg'),
                   cv2.cvtColor(cv2.resize(edge_image, (224, 224)), cv2.COLOR_RGB2BGR))
        
        # Save final processed image (denormalize for saving)
        save_image = (processed * 255).astype(np.uint8)
        cv2.imwrite(os.path.join(output_dir, '4_final.jpg'), 
                   cv2.cvtColor(save_image, cv2.COLOR_RGB2BGR))
        
        print(f"   ✅ Images saved to: {output_dir}")
        print(f"   Files saved:")
        print(f"      - 1_original.jpg")
        print(f"      - 2_mask.jpg (if available)")
        print(f"      - 3_background_removed.jpg")
        print(f"      - 3_edge_detected.jpg")
        print(f"      - 4_final.jpg")
        
        # Clean up
        print("\n🧹 Cleaning up...")
        del bg_remover, edge_detector, preprocessor
        del image, image_rgb, bg_removed, edge_image, processed
        if 'mask' in locals():
            del mask
        if 'edges' in locals():
            del edges
        gc.collect()
        
        print("\n" + "="*50)
        print("✅ Pipeline test successful!")
        print("="*50)
        print("✅ No kernel crash!")
        print(f"✅ Check output images in: {output_dir}")
        
    else:
        print("\n⚠️ No test image found in static/uploads/")
        print("   Creating dummy image for pipeline test...")
        
        # Create a dummy image for testing
        dummy_image = np.zeros((800, 800, 3), dtype=np.uint8)
        dummy_image[:, :] = [100, 150, 200]  # Light blue background
        # Draw a green square in the center (simulating leaf)
        cv2.rectangle(dummy_image, (200, 200), (600, 600), (0, 255, 0), -1)
        # Add some texture
        cv2.rectangle(dummy_image, (250, 250), (550, 550), (0, 200, 0), -1)
        print("   Created dummy image with green square for testing")
        
        image_rgb = cv2.cvtColor(dummy_image, cv2.COLOR_BGR2RGB)
        
        # Process dummy image
        print("\n🔄 Step 1: Background removal...")
        if bg_remover is not None:
            try:
                bg_removed, mask = bg_remover.remove_background(
                    image_rgb, 
                    target_size=(224, 224),
                    max_size=800
                )
                print("   ✅ Background removal complete")
            except Exception as e:
                print(f"   ⚠️ Background removal error: {e}")
                bg_removed = cv2.resize(image_rgb, (224, 224))
        else:
            bg_removed = cv2.resize(image_rgb, (224, 224))
        
        gc.collect()
        
        print("\n🔄 Step 2: Edge detection...")
        if edge_detector is not None:
            try:
                edge_image, edges = edge_detector.detect_edges(bg_removed)
                print("   ✅ Edge detection complete")
            except Exception as e:
                print(f"   ⚠️ Edge detection error: {e}")
                edge_image = bg_removed
        else:
            edge_image = bg_removed
        
        gc.collect()
        
        print("\n🔄 Step 3: Final preprocessing...")
        processed = preprocessor.preprocess_for_model(edge_image)
        print(f"   ✅ Preprocessing complete - shape: {processed.shape}")
        print(f"   Final image range: [{processed.min():.2f}, {processed.max():.2f}]")
        
        print("\n✅ Dummy image pipeline test successful!")
        
except Exception as e:
    print(f"❌ Pipeline test failed: {e}")
    import traceback
    traceback.print_exc()

# %%
# Save preprocessor to file with proper encoding
preprocessor_code = '''
import cv2
import numpy as np
from PIL import Image

class ImagePreprocessor:
    def __init__(self, target_size=(224, 224)):
        self.target_size = target_size
    
    def preprocess_for_model(self, image):
        """
        Final preprocessing for model input
        """
        if isinstance(image, Image.Image):
            image = np.array(image)
        elif hasattr(image, 'cpu'):  # Handle torch tensors
            image = image.cpu().numpy()
        
        # Ensure correct dtype
        if image.dtype != np.uint8:
            if image.max() <= 1.0:
                image = (image * 255).astype(np.uint8)
            else:
                image = image.astype(np.uint8)
        
        # Resize
        image = cv2.resize(image, self.target_size)
        
        # Normalize to [0,1]
        image = image.astype(np.float32) / 255.0
        
        return image
    
    def combine_with_features(self, image, fertilizer_amount, days):
        """
        Combine image with numerical features
        """
        features = np.array([fertilizer_amount, days])
        return image, features
'''

# Ensure utils directory exists
utils_dir = os.path.join(os.getcwd(), 'utils')
os.makedirs(utils_dir, exist_ok=True)

# Save with utf-8 encoding
with open(os.path.join(utils_dir, 'preprocessing.py'), 'w', encoding='utf-8') as f:
    f.write(preprocessor_code)
print(f"\n✅ Saved ImagePreprocessor to {os.path.join(utils_dir, 'preprocessing.py')}")

# %%
# Quick verification (no display)
print("\n" + "="*50)
print("Verifying saved preprocessor")
print("="*50)

try:
    # Import the saved class
    sys.path.append(utils_dir)
    from preprocessing import ImagePreprocessor
    
    # Test
    test_preprocessor = ImagePreprocessor()
    dummy_image = np.zeros((100, 100, 3), dtype=np.uint8)
    result = test_preprocessor.preprocess_for_model(dummy_image)
    
    print(f"✅ Preprocessor verification successful!")
    print(f"   Output shape: {result.shape}")
    print(f"   Output dtype: {result.dtype}")
    print(f"   Output range: [{result.min():.2f}, {result.max():.2f}]")
    
except Exception as e:
    print(f"❌ Verification failed: {e}")

print("\n" + "="*50)
print("🎉 Preprocessing pipeline setup complete!")
print("="*50)
print("\n📝 NOTE: Images were saved to 'test_output' folder instead of displaying")
print("   This prevents kernel death from matplotlib rendering")

Testing Complete Preprocessing Pipeline
✅ BackgroundRemover imported
✅ EdgeDetector imported

🔄 Initializing portable BackgroundRemover...
✅ U2Net source already present
✅ BackgroundRemover initialized with U2Net model
✅ BackgroundRemover initialized

🔄 Processing image: C:\Users\HP\NitroSense-AI\static\uploads\2888723a-fd6f-4471-b7e5-692727bd5692_668bc6dbe4b48.jpg
📏 Image resized to 800x434 for processing

🔄 Step 1: Background removal...
   ✅ Background removal complete
   Background removed shape: (224, 224, 3)
   Mask shape: (224, 224)

🔄 Step 2: Edge detection...
   ✅ Edge detection complete
   Edge detected shape: (224, 224, 3)

🔄 Step 3: Final preprocessing...
   ✅ Preprocessing complete - shape: (224, 224, 3)
   Final image range: [0.00, 1.00]
   Final image dtype: float32

💾 Saving results to verify...
   ✅ Images saved to: C:\Users\HP\NitroSense-AI\test_output
   Files saved:
      - 1_original.jpg
      - 2_mask.jpg (if available)
      - 3_background_removed.jpg
      - 3_ed